In [30]:
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
%matplotlib inline

# Cxt A

In [31]:
import onep

# Load the pickle file
with open('/Users/suthardr/Desktop/collection_cxta_allmice.pkl', 'rb') as file:
    collection_fc = pickle.load(file)

# Define animals for each group
mouse_ids = ['astroF3', 'astroF5', 'astroF6', 'astroF7', 'astroF9', 'astroF10',
             'astroM3', 'astroM4', 'astroM5', 'astroM9', 'astroM10']

In [32]:
#Grab your mice and take accepted traces - convert to numpy arrays within mouse_id variable names
for mouse_id in mouse_ids:
    try:
        traces = collection_fc.animals[mouse_id].accepted_traces.to_numpy()
        globals()[mouse_id] = traces[:3303, :]
        print(f"Created variable {mouse_id} with shape {globals()[mouse_id].shape}")
    except Exception as e:
        print(f"Error loading {mouse_id}: {e}")

Created variable astroF3 with shape (3303, 98)
Created variable astroF5 with shape (3303, 77)
Created variable astroF6 with shape (3303, 35)
Created variable astroF7 with shape (3303, 32)
Created variable astroF9 with shape (3303, 244)
Created variable astroF10 with shape (3303, 55)
Created variable astroM3 with shape (3303, 199)
Created variable astroM4 with shape (3303, 300)
Created variable astroM5 with shape (3303, 200)
Created variable astroM9 with shape (3303, 79)
Created variable astroM10 with shape (3303, 52)


In [33]:
def run_cross_validated_analysis(
    animal_id,
    traces,
    timestamps,
    save_path_prefix,
    event_times,                
    window=28,                   
    do_shuffle=True,
    n_shuffles=10000,
    seed=None,
    two_sided=True
):
    """
    Compute cross-validated sequence correlation between two interleaved sets of events
    (odd vs. even indices of `event_times`) for astrocyte traces.

    Parameters
    ----------
    event_times : array-like
        Absolute or relative event times (seconds). Odd = 1st,3rd,5th,... ; Even = 2nd,4th,6th,...
    window : float
        ETA window length in seconds
    """
    def shuffle_control_test(x, y, n_shuffles=1000, seed=None, two_sided=True):
        x = np.asarray(x); y = np.asarray(y)
        if x.shape[0] != y.shape[0]:
            raise ValueError(f"Length mismatch: x={x.shape[0]}, y={y.shape[0]}")
        if not np.all(np.isfinite(x)) or not np.all(np.isfinite(y)):
            raise ValueError("Non-finite values in x or y")
        rng = np.random.default_rng(seed)
        r_obs, p_obs = stats.spearmanr(x, y)
        sh = np.empty(n_shuffles, dtype=float)
        for i in range(n_shuffles):
            y_perm = rng.permutation(y)
            sh[i], _ = stats.spearmanr(x, y_perm)
        if two_sided:
            p_emp = (np.sum(np.abs(sh) >= abs(r_obs)) + 1) / (n_shuffles + 1)
        else:
            if r_obs >= 0:
                p_emp = (np.sum(sh >= r_obs) + 1) / (n_shuffles + 1)
            else:
                p_emp = (np.sum(sh <= r_obs) + 1) / (n_shuffles + 1)
        return {
            "r_obs": float(r_obs),
            "p_obs_parametric": float(p_obs),
            "shuffle_rhos": sh,
            "p_empirical": float(p_emp),
            "null_mean": float(np.mean(sh)),
            "null_std": float(np.std(sh, ddof=1)),
            "z_obs_vs_null": float((r_obs - np.mean(sh)) / (np.std(sh, ddof=1) + 1e-12)),
        }

    def plot_shuffle_null(shuffle_rhos, r_obs, figsize=(3.5, 3.5), dpi=600):
        fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
        ax.hist(shuffle_rhos, bins=30, density=True)
        ax.axvline(r_obs, linestyle='--')
        ax.set_xlabel('Spearman ρ')
        ax.set_ylabel('Density')
        ax.set_title('Shuffle null (permute Y across cells)')
        return fig, ax

    plt.rcParams.update({'font.size': 8, 'font.family': 'Arial'})

    # ---- Parse and split events into odd/even ----
    event_times = np.asarray(event_times, dtype=float)
    if event_times.ndim != 1 or event_times.size < 2:
        raise ValueError("`event_times` must be a 1D array with at least two times.")

    odd_times  = event_times[0::2].tolist()
    even_times = event_times[1::2].tolist()
    if len(odd_times) == 0 or len(even_times) == 0:
        raise ValueError("Need at least one odd and one even event time.")

    # ---- Compute ETAs for odd and even events ----
    # onep.eta_individual_cells expects a list-of-lists (per your original [[120, 240]] usage),
    # so we wrap the times in a list.
    whole_eta_odd,  time_vec_odd  = onep.eta_individual_cells(traces.T, timestamps, [odd_times],  window=window)
    whole_eta_even, time_vec_even = onep.eta_individual_cells(traces.T, timestamps, [even_times], window=window)

    # ---- Align to time >= 0 (replaces magic index 140) ----
    # We assume time vectors are identical for odd/even; use odd as reference.
    t0_idx = int(np.searchsorted(time_vec_odd, 0.0, side="left"))
    if t0_idx <= 0 or t0_idx >= len(time_vec_odd):
        # Fallback: if 0 isn't inside the window, keep everything (rare, but safer than crashing)
        t0_idx = 0
    time_post0 = time_vec_odd[t0_idx:]

    # ---- Z-score each cell using only the post-0 segment, like your original ----
    adjusted_odd  = stats.zscore(whole_eta_odd[:,  t0_idx:], axis=1)
    adjusted_even = stats.zscore(whole_eta_even[:, t0_idx:], axis=1)

    # ---- Sort arrays by their own maxima ----
    sorted_odd,  idxs_odd  = onep.maxsort(adjusted_odd)
    sorted_even, idxs_even = onep.maxsort(adjusted_even)

    # y-ticks for heatmaps
    first_last = [0, adjusted_odd.shape[0]]

    # --- Heatmaps ---
    fig, (ax1, ax2) = plt.subplots(1, 2, sharey=True, figsize=(7, 5), dpi=600)
    sns.set(style="ticks", font="Arial")

    # Left: odd, sorted by odd
    sns.heatmap(adjusted_odd[idxs_odd], ax=ax1, cmap='mako', cbar=False,
                xticklabels=True, rasterized=True, yticklabels=True, vmin=-4, vmax=4)
    ax1.tick_params(axis='both', which='major', labelsize=8)
    ax1.set_xticks(np.linspace(0, len(time_post0), 5))
    ax1.set_xticklabels(np.linspace(np.min(time_post0), np.max(time_post0), 5).astype(int), rotation=0)
    ax1.set_yticks(first_last)
    ax1.set_yticklabels([first_last[0], first_last[1]], rotation=0)
    ax1.set_title('Odd events (sorted by Odd)', fontsize=10, pad=10)
    ax1.set_ylabel('Astrocyte #', fontsize=10)
    ax1.set_xlabel('Time (s)', fontsize=10)
    sns.despine(ax=ax1, left=True, bottom=True)

    # Right: odd, sorted by even (cross-sorting)
    # FIX: use adjusted_even's sort order (idxs_even) to reorder the odd matrix.
    sns.heatmap(adjusted_odd[idxs_even], ax=ax2, cmap='mako', cbar=True,
                cbar_kws={'label': 'Z-score'}, xticklabels=True, rasterized=True, yticklabels=True, vmin=-4, vmax=4)
    ax2.tick_params(axis='both', which='major', labelsize=8)
    ax2.set_xticks(np.linspace(0, len(time_post0), 5))
    ax2.set_xticklabels(np.linspace(np.min(time_post0), np.max(time_post0), 5).astype(int), rotation=0)
    ax2.set_yticks(first_last)
    ax2.set_yticklabels([first_last[0], first_last[1]], rotation=0)
    ax2.set_title('Odd events (sorted by Even)', fontsize=10, pad=10)
    ax2.set_xlabel('Time (s)', fontsize=10)
    sns.despine(ax=ax2, left=True, bottom=True)

    plt.tight_layout(pad=3)
    heatmap_fp = f"{save_path_prefix}_heatmaps.svg"
    fig.savefig(heatmap_fp, dpi=600, bbox_inches='tight')
    plt.close(fig)

    # --- Argmaxes ---
    max_idxs_odd = np.argmax(sorted_odd, axis=1)
    argmax_odd = time_post0[max_idxs_odd]

    odd_sorted_by_even = adjusted_odd[idxs_even]
    sorted_idx = np.argmax(odd_sorted_by_even, axis=1)
    argmax_odd_sorted_by_even = time_post0[sorted_idx]

    # --- Spearman correlation ---
    rho_s, p_s = stats.spearmanr(argmax_odd, argmax_odd_sorted_by_even)
    print(f"{animal_id} - Spearman rho: {rho_s:.4f}, p-value: {p_s:.4e}")

    # --- Shuffle control (optional) ---
    shuffle_fp = None
    shuffle_empirical_p = None
    shuffle_null_mean = None
    shuffle_null_std = None
    z_obs_vs_null = None

    if do_shuffle:
        if seed is None:
            seed = hash(str(animal_id)) & 0xFFFFFFFF
        shres = shuffle_control_test(
            argmax_odd, argmax_odd_sorted_by_even,
            n_shuffles=n_shuffles, seed=seed, two_sided=two_sided
        )
        print(f"{animal_id} - shuffle null: mean={shres['null_mean']:.4f}, "
              f"sd={shres['null_std']:.4f}, empirical p={shres['p_empirical']:.4g}, "
              f"z={shres['z_obs_vs_null']:.2f}")
        fig_shuf, ax_shuf = plot_shuffle_null(shres["shuffle_rhos"], shres["r_obs"])
        shuffle_fp = f"{save_path_prefix}_shuffle_null.svg"
        fig_shuf.savefig(shuffle_fp, dpi=600, bbox_inches='tight')
        plt.close(fig_shuf)

        shuffle_empirical_p = shres["p_empirical"]
        shuffle_null_mean = shres["null_mean"]
        shuffle_null_std = shres["null_std"]
        z_obs_vs_null = shres["z_obs_vs_null"]

    # --- Regression data (clean) ---
    df = pd.DataFrame({
        'Argmax Odd sorted Odd':  argmax_odd,
        'Argmax Odd sorted Even': argmax_odd_sorted_by_even
    })
    x_name = 'Argmax Odd sorted Odd'
    y_name = 'Argmax Odd sorted Even'
    _df = df[[x_name, y_name]].copy()
    _df = _df[np.isfinite(_df[x_name]) & np.isfinite(_df[y_name])]
    x = _df[x_name].to_numpy()
    y = _df[y_name].to_numpy()
    X = sm.add_constant(x)

    # --- Fit RLM ---
    model = sm.RLM(y, X).fit()

    # ---- R^2 variants ----
    y_pred = np.asarray(model.fittedvalues)
    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    r2_var = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

    # guard for degenerate correlation
    try:
        r2_corr = float(np.corrcoef(y, y_pred)[0, 1] ** 2)
    except Exception:
        r2_corr = np.nan

    rho_rlm = model.model.M.rho
    scale = getattr(model, "scale", 1.0) or 1.0
    resid_fit_scaled = (y - y_pred) / scale
    resid_null_scaled = (y - float(np.mean(y))) / scale
    loss_fit = float(np.sum(rho_rlm(resid_fit_scaled)))
    loss_null = float(np.sum(rho_rlm(resid_null_scaled)))
    r2_pseudo_robust = 1 - (loss_fit / loss_null) if loss_null > 0 else np.nan

    # ---- Extract flattened params ----
    beta  = np.asarray(model.params)        # [intercept, slope]
    pvals = np.asarray(model.pvalues)       # [p_intercept, p_slope]
    intercept = float(beta[0]) if beta.size > 0 else np.nan
    slope     = float(beta[1]) if beta.size > 1 else np.nan
    p_value_bias  = float(pvals[0]) if pvals.size > 0 else np.nan
    p_value_slope = float(pvals[1]) if pvals.size > 1 else np.nan

    # --- Plot: scatter + RLM line + 95% CI (bootstrap with fallback) ---
    fig, ax = plt.subplots(figsize=(3.5, 3.5), dpi=600)
    sns.set(style="ticks", font="Arial")
    sns.scatterplot(data=_df, x=x_name, y=y_name, color='#231f20', ax=ax)

    x_grid = np.linspace(x.min(), x.max(), 200)
    X_grid = np.column_stack([np.ones_like(x_grid), x_grid])
    y_hat = X_grid @ np.asarray(model.params)
    ax.plot(x_grid, y_hat, color='#231f20', lw=2, label='RLM fit')

    n_boot = 1000
    min_ok = 100
    rng = np.random.default_rng(12345)
    y_boot = np.full((n_boot, x_grid.size), np.nan, dtype=float)

    for b in range(n_boot):
        idx = rng.integers(0, len(_df), size=len(_df))
        xb = x[idx]; yb = y[idx]
        Xb = sm.add_constant(xb)
        try:
            mb = sm.RLM(yb, Xb).fit()
            y_boot[b, :] = X_grid @ np.asarray(mb.params)
        except Exception:
            pass

    counts = np.sum(np.isfinite(y_boot), axis=0)
    lower = np.full_like(y_hat, np.nan)
    upper = np.full_like(y_hat, np.nan)
    ok = counts >= min_ok
    if np.any(ok):
        lower[ok] = np.nanpercentile(y_boot[:, ok], 2.5, axis=0)
        upper[ok] = np.nanpercentile(y_boot[:, ok], 97.5, axis=0)

    if np.any(~ok):
        covb = np.asarray(model.cov_params())
        se_mean = np.sqrt(np.sum(X_grid @ covb * X_grid, axis=1))
        z = stats.norm.ppf(0.975)
        lower_analytic = y_hat - z * se_mean
        upper_analytic = y_hat + z * se_mean
        lower[~ok] = lower_analytic[~ok]
        upper[~ok] = upper_analytic[~ok]

    ax.fill_between(x_grid, lower, upper, alpha=0.2, label='95% CI')
    ax.set_xlabel('Argmax Odd Sorted Odd (s)', fontsize=10, fontname="Arial")
    ax.set_ylabel('Argmax Odd Sorted Even (s)', fontsize=10, fontname='Arial')
    ax.tick_params(labelsize=8)
    ax.set_ylim(0, window)   # keep your 0–25 style if window=28; adjust as you like
    ax.set_xlim(0, window)
    sns.despine(ax=ax)
    ax.legend(frameon=False)

    scatter_fp = f"{save_path_prefix}_scatter.svg"
    fig.savefig(scatter_fp, dpi=600, bbox_inches='tight')
    plt.close(fig)

    # ---- Return flattened summary ----
    return {
        "animal": animal_id,
        "spearman_rho": float(rho_s),
        "spearman_p": float(p_s),
        "slope": slope,
        "intercept": intercept,
        "p_value_slope": p_value_slope,
        "p_value_bias": p_value_bias,
        "r2_var": float(r2_var),
        "r2_corr": float(r2_corr) if np.isfinite(r2_corr) else np.nan,
        "r2_pseudo_robust": float(r2_pseudo_robust) if np.isfinite(r2_pseudo_robust) else np.nan,
        "heatmap_fp": heatmap_fp,
        "scatter_fp": scatter_fp,
        "shuffle_hist_fp": shuffle_fp,
        "shuffle_empirical_p": shuffle_empirical_p,
        "shuffle_null_mean": shuffle_null_mean,
        "shuffle_null_std": shuffle_null_std,
        "shuffle_z": z_obs_vs_null,
    }

In [34]:
event_times_csv = r"/Volumes/rkc_ramirezlab/Home/rsenne/dCA1_Clean_Data/revision_analysis/Figure3_Recall/num_events_Fig3/event_times_cxta.csv"
event_times_df = pd.read_csv(event_times_csv)

# Normalize headers (strip spaces) and build column->times mapping
event_times_df.rename(columns={c: c.strip() for c in event_times_df.columns}, inplace=True)

col_to_times = {}
for col in event_times_df.columns:
    s = pd.to_numeric(event_times_df[col], errors='coerce')  # blank cells -> NaN
    times = s.dropna().astype(float).to_numpy()
    # sort & de-duplicate (keep chronological order if CSV already sorted)
    times = np.unique(np.round(times, 9))  # rounding to avoid fp dupes like 10 vs 9.999999999
    col_to_times[col] = times

def pick_column_for_mouse(mouse_id, candidates):
    """
    Choose the best matching column for a mouse_id.
    Priority: startswith(mouse_id) AND contains '_cxta' > startswith(mouse_id) > contains(mouse_id)
    """
    lid = mouse_id.lower()
    # tier 1: startswith + contains _cxta
    t1 = [c for c in candidates if c.lower().startswith(lid) and '_cxta' in c.lower()]
    if t1: return t1[0]
    # tier 2: startswith
    t2 = [c for c in candidates if c.lower().startswith(lid)]
    if t2: return t2[0]
    # tier 3: contains
    t3 = [c for c in candidates if lid in c.lower()]
    if t3: return t3[0]
    return None


In [35]:
import os

save_dir = '/Users/suthardr/Desktop/Revision/fig3/A'
os.makedirs(save_dir, exist_ok=True)

trace_lookup = {mouse: globals()[mouse] for mouse in mouse_ids}
all_cols = list(col_to_times.keys())

all_results = []
skipped = []

for mouse in mouse_ids:
    print(f"Processing {mouse}...")

    # 1) traces (time x cells)
    traces = trace_lookup.get(mouse, None)
    if traces is None:
        print(f"  [WARN] No traces for {mouse}. Skipping.")
        skipped.append((mouse, "no_traces"))
        continue

    # 2) timestamps (trimmed to match your 3303 rows)
    try:
        # You used capital 'T' earlier
        timestamps = np.asarray(collection_fc.animals[mouse].Timestamps[:traces.shape[0]], dtype=float).ravel()
    except Exception as e:
        print(f"  [WARN] No timestamps for {mouse}: {e}. Skipping.")
        skipped.append((mouse, "no_timestamps"))
        continue

    # sanity align (in case lengths drift)
    T = min(traces.shape[0], timestamps.shape[0])
    traces = traces[:T, :]
    timestamps = timestamps[:T]

    # 3) find the CSV column that belongs to this mouse
    # prefer exact "<mouse>_cxta" if present; else use your matcher
    exact_col = f"{mouse}_cxta"
    if exact_col in col_to_times:
        col = exact_col
    else:
        col = pick_column_for_mouse(mouse, all_cols)

    if col is None:
        print(f"  [WARN] No CSV column matched {mouse}. Skipping.")
        skipped.append((mouse, "no_column"))
        continue

    # 4) load & clip event times to the session range
    ev_all = np.asarray(col_to_times[col], dtype=float)
    tmin, tmax = float(np.nanmin(timestamps)), float(np.nanmax(timestamps))
    ev = ev_all[(ev_all >= tmin) & (ev_all <= tmax)]

    # need at least 2 events (so we have odd/even)
    if ev.size < 2:
        print(f"  [WARN] {mouse}: only {ev.size} event(s) in [{tmin:.2f}, {tmax:.2f}] (col={col}). Skipping.")
        skipped.append((mouse, "too_few_events"))
        continue

    # 5) run analysis (odd vs even handled inside)
    try:
        res = run_cross_validated_analysis(
            animal_id=mouse,
            traces=traces,                 # (time x cells); function uses traces.T internally
            timestamps=timestamps,         # 1D seconds
            save_path_prefix=f"{save_dir}/FC_crossvalled_{mouse}",
            event_times=ev,                # <<< key addition
            window=28,
            do_shuffle=True,
            n_shuffles=10000,
            seed=None,
            two_sided=True
        )
        all_results.append(res)
        print(f"  [OK] rho={res['spearman_rho']:.3f}, p={res['spearman_p']:.3g}, n_events={ev.size}")
    except Exception as e:
        print(f"  [ERROR] {mouse}: {e}")
        skipped.append((mouse, "exception"))

# 6) summarize
df_all_results = pd.DataFrame(all_results)
if 'animal' in df_all_results.columns:
    df_all_results = df_all_results[['animal'] + [c for c in df_all_results.columns if c != 'animal']]

print(df_all_results)
out_csv = f'{save_dir}/crossval_summary_results_cxta.csv'
df_all_results.to_csv(out_csv, index=False)
print(f"Saved: {out_csv}")
if skipped:
    print("Skipped:", skipped)


Processing astroF3...
astroF3 - Spearman rho: 0.2992, p-value: 2.7605e-03
astroF3 - shuffle null: mean=0.0029, sd=0.1022, empirical p=0.0028, z=2.90
  [OK] rho=0.299, p=0.00276, n_events=3
Processing astroF5...
astroF5 - Spearman rho: 0.2930, p-value: 9.7092e-03
astroF5 - shuffle null: mean=0.0003, sd=0.1140, empirical p=0.0104, z=2.57
  [OK] rho=0.293, p=0.00971, n_events=3
Processing astroF6...
astroF6 - Spearman rho: 0.3144, p-value: 6.5826e-02
astroF6 - shuffle null: mean=0.0005, sd=0.1710, empirical p=0.06579, z=1.84
  [OK] rho=0.314, p=0.0658, n_events=2
Processing astroF7...
astroF7 - Spearman rho: 0.1951, p-value: 2.8450e-01
astroF7 - shuffle null: mean=0.0025, sd=0.1809, empirical p=0.2906, z=1.06
  [OK] rho=0.195, p=0.284, n_events=2
Processing astroF9...
astroF9 - Spearman rho: 0.4298, p-value: 2.1629e-12
astroF9 - shuffle null: mean=-0.0009, sd=0.0634, empirical p=9.999e-05, z=6.79
  [OK] rho=0.430, p=2.16e-12, n_events=4
Processing astroF10...
astroF10 - Spearman rho: 0.18

# Cxt B

In [36]:
# Load the pickle file
with open('/Users/suthardr/Desktop/collection_cxtb_allmice.pkl', 'rb') as file:
    collection_fc = pickle.load(file)

# Define animals for each group
mouse_ids = ['astroF3', 'astroF5', 'astroF6', 'astroF7','astroF9', 'astroF10',
             'astroM6', 'astroM7', 'astroM8', 'astroM10']

#Grab your mice and take accepted traces - convert to numpy arrays within mouse_id variable names
for mouse_id in mouse_ids:
    try:
        traces = collection_fc.animals[mouse_id].accepted_traces.to_numpy()
        globals()[mouse_id] = traces[:3303, :]
        print(f"Created variable {mouse_id} with shape {globals()[mouse_id].shape}")
    except Exception as e:
        print(f"Error loading {mouse_id}: {e}")

Created variable astroF3 with shape (3303, 55)
Created variable astroF5 with shape (3303, 26)
Created variable astroF6 with shape (3303, 15)
Created variable astroF7 with shape (3303, 12)
Created variable astroF9 with shape (3302, 161)
Created variable astroF10 with shape (3303, 54)
Created variable astroM6 with shape (3303, 134)
Created variable astroM7 with shape (3303, 144)
Created variable astroM8 with shape (3303, 227)
Created variable astroM10 with shape (3302, 96)


In [37]:
event_times_csvB = r"/Volumes/rkc_ramirezlab/Home/rsenne/dCA1_Clean_Data/revision_analysis/Figure3_Recall/num_events_Fig3/event_times_cxtb.csv"
event_times_dfB = pd.read_csv(event_times_csvB)

# Normalize headers (strip spaces) and build column->times mapping
event_times_dfB.rename(columns={c: c.strip() for c in event_times_dfB.columns}, inplace=True)

col_to_times = {}
for col in event_times_dfB.columns:
    s = pd.to_numeric(event_times_dfB[col], errors='coerce')  # blank cells -> NaN
    times = s.dropna().astype(float).to_numpy()
    # sort & de-duplicate (keep chronological order if CSV already sorted)
    times = np.unique(np.round(times, 9))  # rounding to avoid fp dupes like 10 vs 9.999999999
    col_to_times[col] = times

def pick_column_for_mouse(mouse_id, candidates):
    """
    Choose the best matching column for a mouse_id.
    Priority: startswith(mouse_id) AND contains '_cxta' > startswith(mouse_id) > contains(mouse_id)
    """
    lid = mouse_id.lower()
    # tier 1: startswith + contains _cxta
    t1 = [c for c in candidates if c.lower().startswith(lid) and '_cxta' in c.lower()]
    if t1: return t1[0]
    # tier 2: startswith
    t2 = [c for c in candidates if c.lower().startswith(lid)]
    if t2: return t2[0]
    # tier 3: contains
    t3 = [c for c in candidates if lid in c.lower()]
    if t3: return t3[0]
    return None

In [38]:
save_dir = '/Users/suthardr/Desktop/Revision/fig3/B'
os.makedirs(save_dir, exist_ok=True)

trace_lookup = {mouse: globals()[mouse] for mouse in mouse_ids}
all_cols = list(col_to_times.keys())

all_results = []
skipped = []

for mouse in mouse_ids:
    print(f"Processing {mouse}...")

    # 1) traces (time x cells)
    traces = trace_lookup.get(mouse, None)
    if traces is None:
        print(f"  [WARN] No traces for {mouse}. Skipping.")
        skipped.append((mouse, "no_traces"))
        continue

    # 2) timestamps (trimmed to match your 3303 rows)
    try:
        # You used capital 'T' earlier
        timestamps = np.asarray(collection_fc.animals[mouse].Timestamps[:traces.shape[0]], dtype=float).ravel()
    except Exception as e:
        print(f"  [WARN] No timestamps for {mouse}: {e}. Skipping.")
        skipped.append((mouse, "no_timestamps"))
        continue

    # sanity align (in case lengths drift)
    T = min(traces.shape[0], timestamps.shape[0])
    traces = traces[:T, :]
    timestamps = timestamps[:T]

    # 3) find the CSV column that belongs to this mouse
    # prefer exact "<mouse>_cxta" if present; else use your matcher
    exact_col = f"{mouse}_cxtb"
    if exact_col in col_to_times:
        col = exact_col
    else:
        col = pick_column_for_mouse(mouse, all_cols)

    if col is None:
        print(f"  [WARN] No CSV column matched {mouse}. Skipping.")
        skipped.append((mouse, "no_column"))
        continue

    # 4) load & clip event times to the session range
    ev_all = np.asarray(col_to_times[col], dtype=float)
    tmin, tmax = float(np.nanmin(timestamps)), float(np.nanmax(timestamps))
    ev = ev_all[(ev_all >= tmin) & (ev_all <= tmax)]

    # need at least 2 events (so we have odd/even)
    if ev.size < 2:
        print(f"  [WARN] {mouse}: only {ev.size} event(s) in [{tmin:.2f}, {tmax:.2f}] (col={col}). Skipping.")
        skipped.append((mouse, "too_few_events"))
        continue

    # 5) run analysis (odd vs even handled inside)
    try:
        res = run_cross_validated_analysis(
            animal_id=mouse,
            traces=traces,                 # (time x cells); function uses traces.T internally
            timestamps=timestamps,         # 1D seconds
            save_path_prefix=f"{save_dir}/FC_crossvalled_{mouse}",
            event_times=ev,                # <<< key addition
            window=28,
            do_shuffle=True,
            n_shuffles=10000,
            seed=None,
            two_sided=True
        )
        all_results.append(res)
        print(f"  [OK] rho={res['spearman_rho']:.3f}, p={res['spearman_p']:.3g}, n_events={ev.size}")
    except Exception as e:
        print(f"  [ERROR] {mouse}: {e}")
        skipped.append((mouse, "exception"))

# 6) summarize
df_all_results = pd.DataFrame(all_results)
if 'animal' in df_all_results.columns:
    df_all_results = df_all_results[['animal'] + [c for c in df_all_results.columns if c != 'animal']]

print(df_all_results)
out_csv = f'{save_dir}/crossval_summary_results_cxtb.csv'
df_all_results.to_csv(out_csv, index=False)
print(f"Saved: {out_csv}")
if skipped:
    print("Skipped:", skipped)


Processing astroF3...
astroF3 - Spearman rho: -0.0521, p-value: 7.0571e-01
astroF3 - shuffle null: mean=0.0013, sd=0.1366, empirical p=0.7051, z=-0.39
  [OK] rho=-0.052, p=0.706, n_events=3
Processing astroF5...
  [WARN] astroF5: only 1 event(s) in [0.00, 330.20] (col=astroF5_cxtb). Skipping.
Processing astroF6...
  [WARN] astroF6: only 1 event(s) in [0.00, 329.94] (col=astroF6_cxtb). Skipping.
Processing astroF7...
astroF7 - Spearman rho: 0.1329, p-value: 6.8060e-01
astroF7 - shuffle null: mean=0.0006, sd=0.3023, empirical p=0.6846, z=0.44
  [OK] rho=0.133, p=0.681, n_events=2
Processing astroF9...
astroF9 - Spearman rho: 0.2696, p-value: 5.4438e-04
astroF9 - shuffle null: mean=-0.0015, sd=0.0785, empirical p=0.0004, z=3.45
  [OK] rho=0.270, p=0.000544, n_events=5
Processing astroF10...
  [WARN] astroF10: only 1 event(s) in [0.00, 329.94] (col=astroF10_cxtb). Skipping.
Processing astroM6...
astroM6 - Spearman rho: 0.1542, p-value: 7.5296e-02
astroM6 - shuffle null: mean=0.0003, sd=0.0

In [39]:
def compare_rho_vals(cxta_fp, cxtb_fp):
    # read in both csvs
    cxta = pd.read_csv(cxta_fp)
    cxtb = pd.read_csv(cxtb_fp)

    # extract columns we need from both (e.g., animal, spearman_rho, etc.)
    cxta = cxta[["animal", "spearman_rho", "shuffle_null_mean", "shuffle_null_std", "shuffle_z"]]
    cxtb = cxtb[["animal", "spearman_rho", "shuffle_null_mean", "shuffle_null_std", "shuffle_z"]]

    # compute the paired t-test on the spearman_rho within group (e.g., show if greater than an empirical null)
    stata, pa = stats.ttest_rel(cxta["spearman_rho"], cxta["shuffle_null_mean"])
    statb, pb = stats.ttest_rel(cxtb["spearman_rho"], cxtb["shuffle_null_mean"])

    # now compute the differences between groups using the differences in spearman_rho
    diff_a = cxta["spearman_rho"] - cxta["shuffle_null_mean"]
    diff_b = cxtb["spearman_rho"] - cxtb["shuffle_null_mean"]
    stat_diff, p_diff = stats.ttest_ind(diff_a, diff_b, equal_var=False)

    # make statistical plots now yo
    
    return pa, pb, p_diff, stata, statb, stat_diff

compare_rho_vals(
    '/Users/suthardr/Desktop/Revision/fig3/A/crossval_summary_results_cxta.csv',
    '/Users/suthardr/Desktop/Revision/fig3/B/crossval_summary_results_cxtb.csv'
)

(4.668770099483358e-07,
 0.17073396471006191,
 0.03219393040865598,
 11.41469110736903,
 1.555899881148345,
 2.604479215986795)

In [40]:
def boxpaired_two_contexts_vs_shuffle(
    df_a, df_b,
    id_col="animal",
    obs_col="spearman_rho",
    null_col="shuffle_null_mean",
    title="Contexts: ρ vs shuffle",
    ylim=(0, 1),
    palette=("grey", "#B39BC8", "grey", "#C79BC8"),  # A: shuffle, A: obs, B: shuffle, B: obs
    dpi=600,
    save_path=None,
):

    A = df_a[[id_col, obs_col, null_col]].dropna().copy()
    B = df_b[[id_col, obs_col, null_col]].dropna().copy()

    A_long = A.melt(id_vars=id_col, value_vars=[null_col, obs_col],
                    var_name="condition", value_name="rho")
    B_long = B.melt(id_vars=id_col, value_vars=[null_col, obs_col],
                    var_name="condition", value_name="rho")

    name_map = {obs_col: "Observed", null_col: "Shuffle"}
    A_long["condition"] = A_long["condition"].map(name_map)
    B_long["condition"] = B_long["condition"].map(name_map)
    A_long["label"] = A_long["condition"].map({"Shuffle": "A - Shuffle", "Observed": "A - Observed"})
    B_long["label"] = B_long["condition"].map({"Shuffle": "B - Shuffle", "Observed": "B - Observed"})

    long = pd.concat([A_long, B_long], ignore_index=True)

    order = ["A - Shuffle", "A - Observed", "B - Shuffle", "B - Observed"]
    pal = {lab: col for lab, col in zip(order, palette)}
    x_pos = {lab: i for i, lab in enumerate(order)}  # category positions

    plt.rcParams.update({'font.size': 8, 'font.family': 'Arial', 'svg.fonttype': 'none' })
    fig, ax = plt.subplots(figsize=(5.0, 3.5), dpi=dpi)

    # Box
    sns.boxplot(
        data=long, x="label", y="rho",
        order=order, palette=pal, ax=ax,
        boxprops=dict(alpha=0.7),
        medianprops={'color': 'black', 'linewidth': 1},
        whiskerprops={'linewidth': 1, 'color': 'black'},
        capprops={'linewidth': 1, 'color': 'black'},
        width=0.5, showfliers=False
    )

    # Paired lines for A
    A_pairs = A_long.pivot_table(index=id_col, columns="condition", values="rho")
    if {"Shuffle", "Observed"}.issubset(A_pairs.columns):
        for _, row in A_pairs.iterrows():
            ax.plot([x_pos["A - Shuffle"], x_pos["A - Observed"]],
                    [row["Shuffle"], row["Observed"]],
                    color="gray", alpha=0.7, linewidth=1)
            ax.scatter([x_pos["A - Shuffle"], x_pos["A - Observed"]],
                       [row["Shuffle"], row["Observed"]],
                       s=10, color="black", zorder=3)

    # Paired lines for B
    B_pairs = B_long.pivot_table(index=id_col, columns="condition", values="rho")
    if {"Shuffle", "Observed"}.issubset(B_pairs.columns):
        for _, row in B_pairs.iterrows():
            ax.plot([x_pos["B - Shuffle"], x_pos["B - Observed"]],
                    [row["Shuffle"], row["Observed"]],
                    color="gray", alpha=0.7, linewidth=1)
            ax.scatter([x_pos["B - Shuffle"], x_pos["B - Observed"]],
                       [row["Shuffle"], row["Observed"]],
                       s=10, color="black", zorder=3)

    ax.set_xlabel("")
    ax.set_ylabel("Spearman's ρ", labelpad=8)
    if ylim is not None:
        ax.set_ylim(*ylim)
    ax.set_title(title, pad=8)
    sns.despine(ax=ax)
    plt.tight_layout(pad=3)

    if save_path:
        fig.savefig(save_path, dpi=dpi, bbox_inches="tight", format=save_path.split(".")[-1])

    return fig, ax

In [41]:
A_fp = "/Users/suthardr/Desktop/Revision/fig3/A/crossval_summary_results_cxta.csv"
B_fp = "/Users/suthardr/Desktop/Revision/fig3/B/crossval_summary_results_cxtb.csv"
cxta = pd.read_csv(A_fp)
cxtb = pd.read_csv(B_fp)

boxpaired_two_contexts_vs_shuffle(
    cxta, cxtb,
    title="Context A/B: observed vs empirical null",
    save_path="/Users/suthardr/Desktop/Revision/fig3/cxtAB_rho_vs_shuffle.svg",
    ylim=(-0.5, 0.5)
)

/var/folders/5r/7j2h9rcx719_l9q157yrh3nw0000gq/T/ipykernel_84267/2395903486.py:37: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(


(<Figure size 3000x2100 with 1 Axes>,
 <Axes: title={'center': 'Context A/B: observed vs empirical null'}, ylabel="Spearman's ρ">)

In [42]:
def boxpaired_A_vs_B(
    df_a, df_b,
    id_col="animal",
    effect="delta",          # "delta" or "shuffle_z"
    obs_col="spearman_rho",
    null_col="shuffle_null_mean",
    title="A vs B",
    ylabel=None,
    palette=("#8EB1C7", "#C79BC8"),  # A, B
    dpi=600,
    save_path=None,
):
    A = df_a.copy()
    B = df_b.copy()
    if effect == "delta":
        A["_effect"] = A[obs_col] - A[null_col]
        B["_effect"] = B[obs_col] - B[null_col]
        ylab = r"$\Delta \rho$"
    elif effect == "shuffle_z":
        A["_effect"] = B["_effect"] = None  # just to init
        A["_effect"] = A["shuffle_z"]
        B["_effect"] = B["shuffle_z"]
        ylab = "Shuffle z-score"
    else:
        raise ValueError("effect must be 'delta' or 'shuffle_z'")

    A = A[[id_col, "_effect"]].dropna()
    B = B[[id_col, "_effect"]].dropna()

    # Long-form for boxplot
    long = pd.concat([
        A.assign(Context="A").rename(columns={"_effect": "Effect"}),
        B.assign(Context="B").rename(columns={"_effect": "Effect"})
    ], ignore_index=True)

    order = ["A", "B"]
    pal = {"A": palette[0], "B": palette[1]}
    x_pos = {"A": 0, "B": 1}

    # Figure
    plt.rcParams.update({'font.size': 8, 'font.family': 'Arial'})
    fig, ax = plt.subplots(figsize=(3.5, 3.5), dpi=dpi)

    # Boxes
    sns.boxplot(
        data=long, x="Context", y="Effect",
        order=order, palette=pal, ax=ax,
        width=0.6, showfliers=False
    )

    # Dots for ALL animals (paired or not)
    sns.stripplot(
        data=long, x="Context", y="Effect",
        order=order, color="black", size=3,
        jitter=0.0, alpha=0.7, ax=ax, zorder=3
    )

    # Lines only for overlapping animals
    overlap_ids = set(A[id_col]).intersection(set(B[id_col]))
    if overlap_ids:
        merged = (A.rename(columns={"_effect": "_A"})
                    .merge(B.rename(columns={"_effect": "_B"}), on=id_col))
        for _, row in merged.iterrows():
            ax.plot([x_pos["A"], x_pos["B"]], [row["_A"], row["_B"]],
                    color="gray", alpha=0.6, linewidth=1, zorder=2)

    # Labels
    ax.set_xlabel("")
    ax.set_ylabel(ylab if ylabel is None else ylabel, labelpad=8)
    ax.set_title(title, pad=8)
    sns.despine(ax=ax)
    plt.tight_layout(pad=3)

    if save_path:
        fig.savefig(save_path, dpi=dpi, bbox_inches="tight", format=save_path.split(".")[-1])

    return fig, ax

In [43]:
boxpaired_A_vs_B(cxta, cxtb, save_path="/Users/suthardr/Desktop/Revision/fig3/cxtAB_delta_rho.svg")

/var/folders/5r/7j2h9rcx719_l9q157yrh3nw0000gq/T/ipykernel_84267/3195936225.py:45: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(


(<Figure size 2100x2100 with 1 Axes>,
 <Axes: title={'center': 'A vs B'}, ylabel='$\\Delta \\rho$'>)

In [44]:
def boxpaired_two_contexts_vs_shuffle_by_sex(
    df_a, df_b,
    id_col="animal",
    sex_col="sex",
    obs_col="spearman_rho",
    null_col="shuffle_null_mean",
    title="Contexts: ρ vs shuffle",
    ylim=(0, 1),
    palette=("grey", "#B39BC8", "grey", "#C79BC8"),  # A: shuffle, A: obs, B: shuffle, B: obs
    dpi=600,
    save_path=None,
):

    A = df_a[[id_col, sex_col, obs_col, null_col]].dropna().copy()
    B = df_b[[id_col, sex_col, obs_col, null_col]].dropna().copy()

    A_long = A.melt(
        id_vars=[id_col, sex_col],
        value_vars=[null_col, obs_col],
        var_name="condition",
        value_name="rho"
    )
    B_long = B.melt(
        id_vars=[id_col, sex_col],
        value_vars=[null_col, obs_col],
        var_name="condition",
        value_name="rho"
    )

    name_map = {obs_col: "Observed", null_col: "Shuffle"}
    A_long["condition"] = A_long["condition"].map(name_map)
    B_long["condition"] = B_long["condition"].map(name_map)

    A_long["label"] = A_long["condition"].map(
        {"Shuffle": "A - Shuffle", "Observed": "A - Observed"}
    )
    B_long["label"] = B_long["condition"].map(
        {"Shuffle": "B - Shuffle", "Observed": "B - Observed"}
    )

    long = pd.concat([A_long, B_long], ignore_index=True)

    order = ["A - Shuffle", "A - Observed", "B - Shuffle", "B - Observed"]
    pal = {lab: col for lab, col in zip(order, palette)}
    x_pos = {lab: i for i, lab in enumerate(order)}

    sexes = list(long[sex_col].dropna().unique())
    sexes.sort()  # e.g., ['F', 'M']

    plt.rcParams.update({'font.size': 8, 'font.family': 'Arial'})
    n_sex = len(sexes)
    fig, axes = plt.subplots(
        1, n_sex,
        figsize=(5.0 * n_sex, 3.5),
        dpi=dpi,
        sharey=True
    )

    # If only one sex present
    if n_sex == 1:
        axes = [axes]

    for ax, sex in zip(axes, sexes):
        # Subset for this sex
        long_sex = long[long[sex_col] == sex]
        A_long_sex = A_long[A_long[sex_col] == sex]
        B_long_sex = B_long[B_long[sex_col] == sex]

        # Boxes
        sns.boxplot(
            data=long_sex, x="label", y="rho",
            order=order, palette=pal, ax=ax,
            width=0.6, showfliers=False
        )

        # Paired lines for A
        A_pairs = A_long_sex.pivot_table(index=id_col, columns="condition", values="rho")
        if {"Shuffle", "Observed"}.issubset(A_pairs.columns):
            for _, row in A_pairs.iterrows():
                ax.plot(
                    [x_pos["A - Shuffle"], x_pos["A - Observed"]],
                    [row["Shuffle"], row["Observed"]],
                    color="gray", alpha=0.7, linewidth=1
                )
                ax.scatter(
                    [x_pos["A - Shuffle"], x_pos["A - Observed"]],
                    [row["Shuffle"], row["Observed"]],
                    s=10, color="black", zorder=3
                )

        # Paired lines for B
        B_pairs = B_long_sex.pivot_table(index=id_col, columns="condition", values="rho")
        if {"Shuffle", "Observed"}.issubset(B_pairs.columns):
            for _, row in B_pairs.iterrows():
                ax.plot(
                    [x_pos["B - Shuffle"], x_pos["B - Observed"]],
                    [row["Shuffle"], row["Observed"]],
                    color="gray", alpha=0.7, linewidth=1
                )
                ax.scatter(
                    [x_pos["B - Shuffle"], x_pos["B - Observed"]],
                    [row["Shuffle"], row["Observed"]],
                    s=10, color="black", zorder=3
                )

        ax.set_xlabel("")
        ax.set_title(f"{title}\nSex: {sex}", pad=8)
        ax.set_xticklabels(order, rotation=20, ha="right")

        if ylim is not None:
            ax.set_ylim(*ylim)

        sns.despine(ax=ax)

    fig.text(0.04, 0.5, "Spearman's ρ", va="center", rotation="vertical")
    plt.tight_layout(pad=3, rect=(0.06, 0.0, 1.0, 1.0))

    if save_path:
        fig.savefig(save_path, dpi=dpi, bbox_inches="tight",
                    format=save_path.split(".")[-1])

    return fig, axes


In [45]:
boxpaired_two_contexts_vs_shuffle_by_sex(
    cxta, cxtb,
    title="Context A/B: observed vs empirical null",
    save_path="/Users/suthardr/Desktop/Revision/fig3/cxtAB_rho_vs_shuffle_by_sex.svg",
    ylim=(-0.5, 0.5)
)


/var/folders/5r/7j2h9rcx719_l9q157yrh3nw0000gq/T/ipykernel_84267/2546495837.py:70: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
/var/folders/5r/7j2h9rcx719_l9q157yrh3nw0000gq/T/ipykernel_84267/2546495837.py:108: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(order, rotation=20, ha="right")
/var/folders/5r/7j2h9rcx719_l9q157yrh3nw0000gq/T/ipykernel_84267/2546495837.py:70: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
/var/folders/5r/7j2h9rcx719_l9q157yrh3nw0000gq/T/ipykernel_84267/2546495837.py:108: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. afte

(<Figure size 6000x2100 with 2 Axes>,
 array([<Axes: title={'center': 'Context A/B: observed vs empirical null\nSex: F'}, ylabel='rho'>,
        <Axes: title={'center': 'Context A/B: observed vs empirical null\nSex: M'}, ylabel='rho'>],
       dtype=object))

In [46]:
from scipy import stats
import pandas as pd

def compare_rho_vals_by_sex(cxta_fp, cxtb_fp):
    cxta = pd.read_csv(cxta_fp)
    cxtb = pd.read_csv(cxtb_fp)

    # helper to safely pull arrays
    def col(df, name):
        return df[name].astype(float).to_numpy()

    # --- CONTEXT A ---
    A_F = cxta[cxta["sex"] == "F"]
    A_M = cxta[cxta["sex"] == "M"]

    #Paired shuffle vs observed
    tA_F, pA_F = stats.ttest_rel(col(A_F, "spearman_rho"), col(A_F, "shuffle_null_mean"), nan_policy="omit")
    tA_M, pA_M = stats.ttest_rel(col(A_M, "spearman_rho"), col(A_M, "shuffle_null_mean"), nan_policy="omit")

    # Male vs Female (independent)
    tA_sex, pA_sex = stats.ttest_ind(col(A_M, "spearman_rho"), col(A_F, "spearman_rho"),
                                     nan_policy="omit", equal_var=False)

    # --- CONTEXT B ---
    B_F = cxtb[cxtb["sex"] == "F"]
    B_M = cxtb[cxtb["sex"] == "M"]

    tB_F, pB_F = stats.ttest_rel(col(B_F, "spearman_rho"), col(B_F, "shuffle_null_mean"), nan_policy="omit")
    tB_M, pB_M = stats.ttest_rel(col(B_M, "spearman_rho"), col(B_M, "shuffle_null_mean"), nan_policy="omit")

    # Male vs Female (independent)
    tB_sex, pB_sex = stats.ttest_ind(col(B_M, "spearman_rho"), col(B_F, "spearman_rho"),
                                     nan_policy="omit", equal_var=False)

    return pA_F, pA_M, pB_F, pB_M, tA_F, tA_M, tB_F, tB_M, tA_sex, pA_sex, tB_sex, pB_sex


pA_F, pA_M, pB_F, pB_M, tA_F, tA_M, tB_F, tB_M, tA_sex, pA_sex, tB_sex, pB_sex = compare_rho_vals_by_sex(
    "/Users/suthardr/Desktop/Revision/fig3/A/crossval_summary_results_cxta.csv",
    "/Users/suthardr/Desktop/Revision/fig3/B/crossval_summary_results_cxtb.csv"
)

print("Context A Female: p =", pA_F, "t =", tA_F)
print("Context A Male:   p =", pA_M, "t =", tA_M)
print("Context A M vs F: p =", pA_sex, "t =", tA_sex)

print("Context B Female: p =", pB_F, "t =", tB_F)
print("Context B Male:   p =", pB_M, "t =", tB_M)
print("Context B M vs F: p =", pB_sex, "t =", tB_sex)


Context A Female: p = 0.000559177801568476 t = 7.787277809843591
Context A Male:   p = 0.0015124825399196965 t = 7.724262610760485
Context A M vs F: p = 0.6577422969509104 t = 0.4587807162786008
Context B Female: p = 0.34030836723778213 t = 1.241383468594219
Context B Male:   p = 0.4322570489927517 t = 0.90482482850679
Context B M vs F: p = 0.9063177553431244 t = -0.12378218773744822
